In [1]:
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression


In [2]:
lf = pl.scan_csv('data/boat_sales.csv', infer_schema_length=1000)

In [9]:
lf.describe()

statistic,,id,type,boatClass,make,model,year,condition,length_ft,beam_ft,dryWeight_lb,hullMaterial,fuelType,numEngines,totalHP,maxEngineYear,minEngineYear,engineCategory,price,sellerId,city,state,zip,created_date,created_month,created_year
str,f64,f64,str,str,str,str,f64,str,f64,str,str,str,str,f64,str,str,str,str,f64,f64,str,str,str,str,f64,f64
"""count""",18903.0,18903.0,"""18903""","""18903""","""18903""","""18903""",18903.0,"""18903""",18903.0,"""18903""","""18903""","""18903""","""18903""",18903.0,"""18903""","""18903""","""18903""","""18903""",18903.0,18903.0,"""18903""","""18903""","""18903""","""18903""",18903.0,18903.0
"""null_count""",0.0,0.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0,"""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""",0.0,0.0,"""0""","""0""","""0""","""0""",0.0,0.0
"""mean""",9782.042216,6.9473e6,null,null,null,null,2013.145956,null,23.803935,null,null,null,null,1.069513,null,null,null,null,647146.88093,49891.652225,null,null,null,null,6.945511,2018.503465
"""std""",5749.171393,482051.924121,null,null,null,null,10.502989,null,14.613329,null,null,null,null,0.425528,null,null,null,null,7.3096e7,60725.592092,null,null,null,null,3.007115,1.093103
"""min""",1.0,444913.0,"""power""","""power-aft""","""33rd Strike Group""","""""",1910.0,"""new""",1.0,"""0.08""","""100""","""aluminum""","""""",0.0,"""0""","""1938""","""1938""","""""",500.0,1003.0,"""""","""AK""","""""","""2003-02-06""",1.0,2003.0
"""25%""",4784.0,6.895219e6,null,null,null,null,2011.0,null,18.0,null,null,null,null,1.0,null,null,null,null,19265.0,10550.0,null,null,null,null,5.0,2018.0
"""50%""",9603.0,7.061616e6,null,null,null,null,2019.0,null,21.0,null,null,null,null,1.0,null,null,null,null,34195.0,34482.0,null,null,null,null,8.0,2019.0
"""75%""",14719.0,7.179153e6,null,null,null,null,2019.0,null,25.0,null,null,null,null,1.0,null,null,null,null,57830.0,53226.0,null,null,null,null,9.0,2019.0
"""max""",20000.0,7.271336e6,"""unpowered""","""unpowered-tender""","""solo skiff""","""z2400""",2020.0,"""used""",375.0,"""NA""","""NA""","""wood""","""other""",4.0,"""NA""","""NA""","""NA""","""v-drive""",1.0000e10,269557.0,"""wilmington""","""WV""","""NA""","""2019-11-02""",12.0,2019.0


In [4]:
df = lf.collect()
df = df.with_columns(
    pl.col(pl.String)
    .replace(["NA", ""], None)
)
df = df.with_columns(pl.col(['dryWeight_lb', 'totalHP', 'beam_ft']).cast(pl.Float64))
df = df.with_columns(pl.col(['maxEngineYear', 'minEngineYear']).cast(pl.Int64))
df = df.with_columns(pl.col(pl.Float64).fill_null(strategy='mean'))


In [5]:

numeric_columns = ['totalHP', 'length_ft', 'beam_ft', 'dryWeight_lb']
category_columns = ['condition', 'type','hullMaterial', 'fuelType', 'engineCategory', 'boatClass']

In [6]:
df.select(pl.col("price").drop_nulls().quantile(0.90)).item()

102900.0

In [7]:
x_columns = numeric_columns + category_columns

q90 = df.select(pl.col("price").drop_nulls().quantile(0.90)).item()

df = df.filter(pl.col("price") <= q90)
X = df[x_columns]
y = df['price']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ("scaler", MinMaxScaler())
])
category_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_columns),
    ('cat', category_transformer, category_columns)
])

model_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LinearRegression())
])
model_pipeline.fit(X_train, y_train)
score = model_pipeline.score(X_test, y_test)
print(f"R2 score: {score:.4f}")

R2 score: 0.4043
